# Pacman DQN training on Google Colab

Trains the DQN agent of [CL-BZH/Pacman](https://github.com/CL-BZH/Pacman) on a GPU.

**Before running:** *Runtime → Change runtime type → T4 GPU*.

Free Colab sessions can be stopped after a few hours or when idle, so checkpoints are saved to your Google Drive and the training can be resumed in a new session.

Checkpoints (`pacman_dqn_<step>.pth`, `pacman_dqn_final.pth`) and the per-game log
(`pacman_dqn_training_log.csv`) are written to `SAVE_DIR`. To continue training in a new
session, set `RESUME_FROM` to the last checkpoint.

To watch the trained agent play locally, copy the checkpoint into the repository and in
`pacman_play.py` use `DQN_PacmanAgent([env], n_frames=2)` with `agent.load(...)`.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
SAVE_DIR = "/content/drive/MyDrive/pacman_dqn"
import os; os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
import os
REPO_DIR = "/content/Pacman"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !git -C {REPO_DIR} pull
else:
    !git clone https://github.com/CL-BZH/Pacman.git {REPO_DIR}
assert os.path.isdir(REPO_DIR), "Cloning the repository failed: check the error message above."
%cd {REPO_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: training will run on CPU (set TRAIN_FREQ = 2)"
import torch; print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 2. Configuration

In [ ]:
# ---- Training configuration ----------------------------------------------
TOTAL_STEPS = 3_000_000       # environment steps for this session (all envs together)
N_ENVS      = 16
N_FRAMES    = 2               # must match the checkpoint when resuming
BATCH_SIZE  = 128
TRAIN_FREQ  = 1               # 1 network update per move of all environments (use 2 on CPU)
INITIAL_LR  = 2.5e-4
MIN_LR      = 5e-5
SAVE_EVERY  = 250_000         # checkpoint interval (steps)

# To continue a previous session, set the path of a checkpoint (.pth), e.g.
# RESUME_FROM = f"{SAVE_DIR}/pacman_dqn_final.pth"
RESUME_FROM = None

## 3. Training

In [ ]:
import torch
from pacman_training import pacman_conf
from pacman_env import PacmanEnvironment
from pacman_agents import DQN_PacmanAgent

resuming = RESUME_FROM is not None
envs = [PacmanEnvironment(pacman_conf) for _ in range(N_ENVS)]
agent = DQN_PacmanAgent(envs, gamma=0.99, initial_lr=INITIAL_LR if not resuming else MIN_LR * 2,
                        initial_tau=0.005, n_frames=N_FRAMES, batch_size=BATCH_SIZE,
                        buffer_capacity=300_000, n_step=3, double=True,
                        network="dueling", reward_scale=0.1)
if resuming:
    agent.load(RESUME_FROM)
    print("Resumed from", RESUME_FROM)

history = agent.run_episodes(
    total_steps=TOTAL_STEPS, episode_max_length=1000,
    # When resuming, the policy is already trained: explore much less
    epsilon_start=0.10 if resuming else 1.0, epsilon_end=0.02,
    epsilon_fraction=0.05 if resuming else 0.25,
    learning_starts=20_000, train_freq=TRAIN_FREQ, gradient_steps=1,
    min_lr=MIN_LR, log_interval=500, save_interval=SAVE_EVERY,
    save_dir=SAVE_DIR)

## 4. Training curves

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt

log = pd.read_csv(f"{SAVE_DIR}/pacman_dqn_training_log.csv")
w = 1000
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].plot(log["step"], log["reward"].rolling(w).mean()); ax[0].set_title("Reward per game")
ax[1].plot(log["step"], 100 * log["won"].rolling(w).mean(), label="won")
ax[1].plot(log["step"], 100 * log["lost"].rolling(w).mean(), label="lost")
ax[1].set_title("Games (%)"); ax[1].legend()
ax[2].plot(log["step"], log["ghosts_left"].rolling(w).mean()); ax[2].set_title("Ghosts left at the end")
for a in ax: a.set_xlabel("environment steps")
plt.tight_layout(); plt.show()

## 5. Evaluation (greedy policy)

In [ ]:
import numpy as np

def evaluate(checkpoint, n_games=500, max_moves=800):
    env = PacmanEnvironment(pacman_conf)
    player = DQN_PacmanAgent([env], n_frames=N_FRAMES)
    player.load(checkpoint)
    player.epsilon = 0.
    won = lost = aborted = 0
    for _ in range(n_games):
        player.reset_episode()
        moves = 0
        while not env.end and moves < max_moves:
            state = player.image_to_state(player.image_processing(env.rgb_image))
            _, action = player.select_action(state, stockastic=False)
            env.step(action)
            moves += 1
        won += env.end and not env.dead
        lost += env.dead
        aborted += not env.end
    print(f"{checkpoint}: won {100*won/n_games:.1f}%, lost {100*lost/n_games:.1f}%, "
          f"aborted {100*aborted/n_games:.1f}%")

evaluate(f"{SAVE_DIR}/pacman_dqn_final.pth")